# Part 3 — Care AI Conversation

## Background

**Care AI** is a virtual oncology nurse that conducts daily video calls with patients undergoing cancer treatment.  
Its primary objectives are:
1. **Early detection of Adverse Events (AEs)** — through conversation and visual observation
2. **Clinical reasoning** — estimating AE severity (CTCAE grade) and recommending appropriate action
3. **Patient experience** — maintaining a warm, empathetic tone that encourages honest symptom reporting

### Why this matters
In standard clinical trials, patients are only assessed during scheduled hospital visits (every 2-3 weeks).  
Between visits, AEs can go undetected — especially mild ones (Grade 1-2) that patients may not report voluntarily.  
Care AI bridges this gap by talking to patients daily and catching symptoms early, before they escalate.

### This notebook
We evaluate **MedGemma-4B** as the Care AI nurse across **6 prompt strategies** to find the optimal approach.  
The experiment has two parts:
- **Live inference** on a single patient case (Sections 1-5) — to inspect actual model outputs
- **Aggregated evaluation** across 8 test patients (Section 6) — for statistical comparison

### Conversation Structure
Each patient interaction follows a multi-round protocol:
```
Patient  → Nurse (Round 1) → Patient → Nurse (Round 2) → Patient → Nurse (Round 3)
                                                                         ↓
                                                              Final Clinical Assessment
```
In each round, the patient describes symptoms and answers questions, while the nurse acknowledges,  
asks targeted follow-up questions about suspected AEs, and refines its clinical picture.  
After the final round, the nurse produces a structured assessment: detected AEs, estimated CTCAE grades, and a recommended action.

---

This notebook is part of the **MedGemma Clinical Trial Engine** pipeline:

```
Part 1  Visual AE Detection ─── MedGemma 1.5 + MedSigLIP (image → AE classification)
Part 2  Cough Detection ──────── HeAR + 2-Stage Classifier (audio → cough type)
Part 3  Care AI Conversation ── MedGemma-4B as virtual nurse (multi-turn dialogue → AE detection)
    ↓
Part 4  Rule Set Generation ─── 10 biomedical DBs → LLM synthesis → simulation parameters
Part 5  Simulation Pipeline ─── Hazard functions + LLM enrichment → synthetic clinical trial data
    ↓
Part 6  Anti-Hallucination ──── RLFR fine-tuning to reduce fabrication in medical text
Part 7  Doc Agent ──────────────  CRF data → MedWatch 3500A pharmacovigilance reports
```


### Prompt Strategies Under Evaluation

We test 6 system prompt designs that vary in the amount and type of context provided to MedGemma:

| Strategy | What the model receives | Key question it answers |
|---|---|---|
| **Baseline** | Drug name, visual assessment, AE profile | Does a standard prompt work? |
| **Concise** | Same as Baseline + output length constraint | Does brevity improve focus? |
| **Full Context** | + patient personality, mood, interaction tips | Does knowing the patient help? |
| **No Drug Info** | Only visual assessment (no AE profile) | Can the model rely on its own medical knowledge? |
| **Few-Shot** | Full Context + example Q&A for each AE | Do examples improve targeting? |
| **Deployment-Ready** | Drug info + visual assessment + speaking style (no personality data) | **What's realistic in production?** |

In [ ]:
# Run once to install dependencies
!pip install -q torch transformers pandas huggingface-hub

In [1]:
import sys, os, json, re, time
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from IPython.display import display, HTML, Markdown

import os
ROOT = Path("/data2/workspace/ClinicalTrialEngine")
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

MODEL_PATH = os.environ.get("MEDGEMMA_MODEL_ID", "google/medgemma-4b-it")
GPU_ID = int(os.environ.get("GPU_ID", "6"))

## 1. Load MedGemma-4B

In [2]:
device = f"cuda:{GPU_ID}"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map={"": device},
)
model.eval()

def nurse_fn(system_prompt: str, user_prompt: str) -> dict:
    """Generate a structured JSON response from MedGemma given system + user prompts."""
    chat = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, do_sample=True)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(match.group()) if match else {"_raw": raw[:800]}
    except (json.JSONDecodeError, AttributeError):
        return {"_raw": raw[:800]}

print(f"MedGemma-4B loaded on GPU {GPU_ID}")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

MedGemma-4B loaded on GPU 6


## 2. Patient Scenario

We use a simulated patient case from our out-of-distribution test set.  
The model does **not** see the ground-truth AEs — it must detect them through conversation.

| Field | Value |
|---|---|
| Drug regimen | Paclitaxel + Carboplatin + Bevacizumab |
| Indication | Non-small cell lung cancer (NSCLC) |
| Treatment day | Day 32 |
| Patient personality | Stoic minimizer (tends to downplay symptoms) |
| **Ground-truth AEs** | **Constipation (G1), Anxiety (G1), Anaemia (G1)** |

In [3]:
with open("data/training_v3_test_ood/sft_data.jsonl") as f:
    for i, line in enumerate(f):
        if i == 1:
            scenario_raw = json.loads(line)
            break

ctx = scenario_raw["context"]
gt_aes = scenario_raw.get("gt_non_visual_aes", [])

from src.engine.mood import MoodState, compute_interaction_quality

mood = MoodState(persona_type="stoic_minimizer", seed=43)
for dim, val in ctx.get("patient_mood", {}).items():
    if dim in mood.state:
        mood.state[dim] = val
quality = compute_interaction_quality(mood)

scenario = {
    "drug_name": ctx["drug_name"],
    "indication": ctx["indication"],
    "visual_assessment": ctx["visual_assessment"],
    "drug_ae_profile": ctx["drug_ae_profile"],
    "treatment_day": ctx["treatment_day"],
    "gt_non_visual_aes": gt_aes,
    "gt_visual_aes": [],
    "patient_demographics": {"age": 60, "sex": "M"},
    "patient_persona_type": "stoic_minimizer",
    "patient_mood": ctx.get("patient_mood", {}),
}
t1_visible = ctx["patient_said"]

print("Patient greeting:")
print(f"  \"{t1_visible.get('greeting', '')}\"")
print(f"\nReported symptoms:")
for s in t1_visible.get('reported_symptoms', []):
    print(f"  - {s.get('symptom', '')}")
print(f"\nGround-truth AEs (hidden from model):")
for ae in gt_aes:
    print(f"  - {ae['ae_term']} (Grade {ae['grade']})")
print(f"\nVisual assessment findings: {ctx['visual_assessment'].get('findings', [])}")

Patient greeting:
  "Oh, hello again. Another day, another dollar, I suppose."

Reported symptoms:
  - I've been feeling a bit…off. Just generally weak and tired, you know? Not sleeping great either.
  - Also, things aren't moving as smoothly as they should in the bathroom, if you catch my drift. A little uncomfortable.

Ground-truth AEs (hidden from model):
  - constipation (Grade 1)
  - anxiety (Grade 1)
  - anaemia (Grade 1)

Visual assessment findings: []


## 3. Prompt Strategy Comparison — Nurse Responses

Each strategy generates the nurse's **Round 1** response to the same patient greeting.  
The model outputs a structured JSON with:
- `acknowledgment` — empathetic response to the patient
- `questions` — follow-up questions, each targeting a specific suspected AE
- `approach_style` — chosen conversational tone (empathetic/concerned/neutral/urgent)

We measure **GT Hit**: how many of the 3 ground-truth AEs are targeted by the nurse's questions.

In [4]:
from src.experiments.eval_prompt_templates import (
    template_a_current, template_b_concise, template_c_rich_persona,
    template_d_minimal, template_e_fewshot, template_f_realistic,
)

STRATEGIES = {
    "Baseline":         template_a_current,
    "Concise":          template_b_concise,
    "Full Context":     template_c_rich_persona,
    "No Drug Info":     template_d_minimal,
    "Few-Shot":         template_e_fewshot,
    "Deployment-Ready": template_f_realistic,
}

STRATEGY_DESC = {
    "Baseline":         "Drug + visual + AE profile (control)",
    "Concise":          "Same context, constrained output length",
    "Full Context":     "+ patient personality, mood, interaction tips",
    "No Drug Info":     "Visual assessment only — no AE profile",
    "Few-Shot":         "Full context + example questions per AE",
    "Deployment-Ready": "Drug + visual + speaking style (no personality data)",
}

for k, v in STRATEGY_DESC.items():
    print(f"  {k:18s}  {v}")

Logging initialized → /data2/workspace/ClinicalTrialEngine/logs/sim_20260223_084609.log


  Baseline            Drug + visual + AE profile (control)
  Concise             Same context, constrained output length
  Full Context        + patient personality, mood, interaction tips
  No Drug Info        Visual assessment only — no AE profile
  Few-Shot            Full context + example questions per AE
  Deployment-Ready    Drug + visual + speaking style (no personality data)


In [5]:
history = [{**t1_visible, "_role": "patient", "_turn": 1}]

results = []
for strategy_name, tmpl_fn in STRATEGIES.items():
    sys_prompt, usr_prompt = tmpl_fn(scenario, quality, turn=2, history=history)
    t0 = time.time()
    response = nurse_fn(sys_prompt, usr_prompt)
    elapsed = time.time() - t0

    ack = response.get("acknowledgment") or response.get("_raw", "")[:200] or ""
    questions = response.get("questions", [])
    targets = [str(q.get("target_ae", "") or "") for q in questions if isinstance(q, dict)]
    style = response.get("approach_style") or "?"
    concerns = response.get("preliminary_concerns") or response.get("concerns") or []

    results.append({
        "strategy": strategy_name,
        "approach_style": style,
        "acknowledgment": ack[:200] if isinstance(ack, str) else str(ack)[:200],
        "n_questions": len(questions),
        "target_aes": ", ".join(targets),
        "concerns": concerns if isinstance(concerns, list) else [],
        "time_s": round(elapsed, 1),
        "_full_response": response,
    })
    print(f"  {strategy_name:18s}  {len(questions)} questions  targets=[{', '.join(targets)[:60]}]  {elapsed:.1f}s")

print(f"\nAll {len(results)} strategies completed.")

  Baseline            3 questions  targets=[nausea, neuropathy_peripheral, insomnia]  12.1s


  Concise             2 questions  targets=[diarrhoea, nausea]  8.1s


  Full Context        3 questions  targets=[anaemia, peripheral_sensory_neuropathy, insomnia]  11.4s


  No Drug Info        2 questions  targets=[Loss of appetite, nausea, or changes in taste, Peripheral ne]  12.8s


  Few-Shot            2 questions  targets=[anaemia, constipation]  8.3s


  Deployment-Ready    2 questions  targets=[anaemia, constipation]  6.7s

All 6 strategies completed.


### AE Targeting Accuracy

**GT Hit** = how many of the 3 ground-truth AEs (constipation, anxiety, anaemia) the nurse's questions target.

In [6]:
def normalize_ae_term(term: str) -> str:
    return term.strip().lower().replace(" ", "_").replace("-", "_")

gt_terms = {normalize_ae_term(ae["ae_term"]) for ae in gt_aes}

def check_hit(target_str):
    targets = {normalize_ae_term(t.strip()) for t in target_str.split(",") if t.strip()}
    hits = sum(1 for g in gt_terms if any(g in t or t in g for t in targets))
    return f"{hits}/{len(gt_terms)}"

df = pd.DataFrame([{
    "Prompt Strategy": r["strategy"],
    "Tone": r["approach_style"],
    "Questions": r["n_questions"],
    "AEs Targeted": r["target_aes"][:80],
    "GT Hit": check_hit(r["target_aes"]),
    "Latency": f"{r['time_s']}s",
} for r in results])

display(df.style.set_properties(**{"text-align": "left"}))

,Prompt Strategy,Tone,Questions,AEs Targeted,GT Hit,Latency
0,Baseline,empathetic,3,"nausea, neuropathy_peripheral, insomnia",0/3,12.1s
1,Concise,empathetic,2,"diarrhoea, nausea",0/3,8.1s
2,Full Context,empathetic,3,"anaemia, peripheral_sensory_neuropathy, insomnia",1/3,11.4s
3,No Drug Info,empathetic,2,"Loss of appetite, nausea, or changes in taste, Peripheral neuropathy, blurred vi",0/3,12.8s
4,Few-Shot,empathetic,2,"anaemia, constipation",2/3,8.3s
5,Deployment-Ready,empathetic,2,"anaemia, constipation",2/3,6.7s


### Full Nurse Responses

Inspect how each prompt strategy shapes the nurse's conversational style and question targeting.

In [7]:
for r in results:
    print(f"\n{'='*70}")
    print(f"  {r['strategy']} — {STRATEGY_DESC[r['strategy']]}")
    print(f"{'='*70}")
    print(f"  Tone: {r['approach_style']}")
    print(f"  Acknowledgment: {r['acknowledgment'][:250]}")
    print(f"  Questions:")
    for q in r["_full_response"].get("questions", []):
        if isinstance(q, dict):
            print(f"    → [{q.get('target_ae','?')}] {str(q.get('question',''))[:120]}")
    if r['concerns']:
        print(f"  Clinical concerns: {r['concerns']}")


  Baseline — Drug + visual + AE profile (control)
  Tone: empathetic
  Acknowledgment: I understand that you've been feeling a bit off and tired lately, and that things aren't moving as smoothly in the bathroom. It's good you're being proactive about your health.
  Questions:
    → [nausea] How is your appetite? Have you noticed any changes in your stomach, like nausea or constipation?
    → [neuropathy_peripheral] Have you experienced any tingling or numbness in your hands or feet?
    → [insomnia] Are you having any trouble sleeping, or do you feel like you're not getting enough rest?
  Clinical concerns: ["I'm concerned about the potential for side effects like nausea, constipation, or neuropathy, especially with this treatment regimen. It's important to be aware of these possibilities and report any new or worsening symptoms."]

  Concise — Same context, constrained output length
  Tone: empathetic
  Acknowledgment: I'm sorry to hear you're feeling a bit off and having some digest

## 4. Clinical Assessment — AE Detection & Reasoning

After the conversation, the nurse generates a **final clinical assessment**:
- Which AEs it suspects, with estimated CTCAE grade and evidence
- A recommended action on the escalation ladder:
  `no_action → monitor_closely → recommend_conmed → recommend_early_visit → recommend_hospital_visit`

We score this against ground truth:
- **Recall**: fraction of true AEs detected
- **Precision**: fraction of detected AEs that are real
- **F1**: harmonic mean of recall and precision
- **Grade MAE**: mean absolute error in CTCAE grade estimation

In [8]:
from src.experiments.eval_prompt_templates import generate_final_assessment, score_t4_assessment

assessment_results = {}
for r in results:
    resp = r["_full_response"]
    resp["_turn"] = 2
    resp["_role"] = "nurse"
    conversation = [{**t1_visible, "_role": "patient", "_turn": 1}, resp]

    assessment = generate_final_assessment(scenario, conversation, nurse_fn)
    score = score_t4_assessment(assessment, gt_aes)
    assessment_results[r["strategy"]] = {"assessment": assessment, "score": score}

    detected = [d.get('ae_term','') for d in assessment.get('detected_aes',[]) if isinstance(d,dict)]
    s = score
    print(f"  {r['strategy']:18s}  Recall={s['ae_recall']:.2f}  Precision={s['ae_precision']:.2f}  F1={s['ae_f1']:.2f}  GradeMAE={s['grade_mae']:.1f}  → {assessment.get('action','?')}  detected={detected}")

  Baseline            Recall=0.00  Precision=0.00  F1=0.00  GradeMAE=0.0  → ?  detected=[]


  Concise             Recall=0.67  Precision=0.40  F1=0.50  GradeMAE=1.0  → monitor_closely  detected=['nausea', 'constipation', 'insomnia', 'anaemia', 'peripheral_neuropathy']


  Full Context        Recall=0.67  Precision=0.50  F1=0.57  GradeMAE=1.5  → monitor_closely  detected=['nausea', 'constipation', 'anaemia', 'insomnia']


  No Drug Info        Recall=0.00  Precision=0.00  F1=0.00  GradeMAE=0.0  → ?  detected=[]


  Few-Shot            Recall=0.67  Precision=0.50  F1=0.57  GradeMAE=1.5  → monitor_closely  detected=['nausea', 'constipation', 'anaemia', 'insomnia']


  Deployment-Ready    Recall=0.67  Precision=0.50  F1=0.57  GradeMAE=2.0  → monitor_closely  detected=['anaemia', 'constipation', 'insomnia', 'peripheral_neuropathy']


### Detailed Assessment — Top 3 Strategies

Inspecting the clinical reasoning for the three most promising prompt strategies.

In [9]:
gt_labels = ["{} (Grade {})".format(ae["ae_term"], ae["grade"]) for ae in gt_aes]
print("Ground Truth AEs:", gt_labels, "\n")

for strategy in ["Deployment-Ready", "Few-Shot", "Full Context"]:
    data = assessment_results.get(strategy)
    if not data:
        continue
    a, s = data["assessment"], data["score"]
    print("=" * 60)
    print(f"  {strategy}")
    print("=" * 60)
    action_reason = a.get("action_reason", "")[:150]
    print(f"  Recommended action: {a.get('action', '?')}")
    print(f"  Reason: {action_reason}")
    print(f"  Concern level: {a.get('overall_concern_level', '?')}")
    print(f"  Detected AEs:")
    for d in a.get("detected_aes", []):
        if isinstance(d, dict):
            print(f"    • {d.get('ae_term','?')} — estimated Grade {d.get('estimated_grade','?')} (confidence: {d.get('confidence','?')})")
            print(f"      Evidence: {str(d.get('evidence',''))[:120]}")
    missed = a.get('missed_screening', [])
    if missed:
        print(f"  Could not assess: {missed}")
    print(f"  Score: Recall={s['ae_recall']:.2f}  Precision={s['ae_precision']:.2f}  F1={s['ae_f1']:.2f}  GradeMAE={s['grade_mae']:.1f}")
    print()

Ground Truth AEs: ['constipation (Grade 1)', 'anxiety (Grade 1)', 'anaemia (Grade 1)'] 

  Deployment-Ready
  Recommended action: monitor_closely
  Reason: The patient reports symptoms suggestive of anaemia, constipation, and insomnia, all of which are potential adverse events associated with paclitaxel, 
  Concern level: low
  Detected AEs:
    • anaemia — estimated Grade 3 (confidence: medium)
      Evidence: Patient reports feeling 'weak and tired' and 'off', which can be indicative of anaemia.
    • constipation — estimated Grade 3 (confidence: medium)
      Evidence: Patient reports 'things aren't moving as smoothly as they should in the bathroom'. This is a direct symptom of constipat
    • insomnia — estimated Grade 2 (confidence: medium)
      Evidence: Patient reports 'Not sleeping great either.' This is a symptom of insomnia.
    • peripheral_neuropathy — estimated Grade 2 (confidence: low)
      Evidence: Patient reports 'things aren't moving as smoothly as they should in th

## 5. Aggregated Evaluation — 8 Patients × 3 Nurse Rounds

The single-patient demo above shows qualitative outputs. Below, we aggregate results across  
**8 out-of-distribution test patients** with 3 nurse rounds each.

We evaluate two aspects:

**A. During conversation — does the nurse ask the right questions?**
- **% AEs Probed**: Of all ground-truth AEs, what fraction did the nurse ask about during the conversation?
- **Patient Comfort**: Did the patient stay engaged and willing to share? (0-1, based on simulated patient behavior)
- **Overall**: Combined score — high only when the nurse detects AEs *without* alienating the patient

**B. Final clinical assessment — is the nurse's conclusion correct?**
- **Recall**: Of all true AEs, how many did the nurse include in its final report?
- **Precision**: Of all AEs the nurse reported, how many were actually real?
- **F1**: Balanced accuracy (harmonic mean of Recall and Precision)
- **Grade Error**: How far off the estimated CTCAE severity was, on average (0 = perfect)

In [10]:
with open("data/training_v3_test_ood/eval_prompt_templates_baseline.json") as f:
    res = json.load(f)

INTERNAL_TO_DISPLAY = {
    "A_current": "Baseline", "B_concise": "Concise", "C_rich": "Full Context",
    "D_minimal": "No Drug Info", "E_fewshot": "Few-Shot", "F_realistic": "Deployment-Ready",
}

# --- Table A: Conversation quality ---
conv_rows = []
for internal, display_name in INTERNAL_TO_DISPLAY.items():
    r = res[internal]
    prog = r["turn_ae_progression"]
    conv_rows.append({
        "Prompt Strategy": display_name,
        "% AEs Probed": r["avg_ae"],
        "Patient Comfort": r["avg_mood"],
        "Overall": r["avg_pareto"],
        "After Round 1": prog[0], "After Round 2": prog[1], "After Round 3": prog[2],
    })
df_conv = pd.DataFrame(conv_rows).set_index("Prompt Strategy")

display(HTML("<h3>A. During Conversation — Does the nurse ask the right questions?</h3>"))
display(df_conv.style.format("{:.3f}")
    .highlight_max(axis=0, subset=["% AEs Probed", "Overall"], color="#c6efce"))

# --- Table B: Final assessment accuracy ---
assess_rows = []
for internal, display_name in INTERNAL_TO_DISPLAY.items():
    r = res[internal]
    assess_rows.append({
        "Prompt Strategy": display_name,
        "Recall": r["t4_recall"],
        "Precision": r["t4_precision"],
        "F1": r["t4_f1"],
        "Grade Error": r["t4_grade_mae"],
    })
df_assess = pd.DataFrame(assess_rows).set_index("Prompt Strategy")

display(HTML("<h3>B. Final Assessment — Is the nurse's conclusion correct?</h3>"))
display(df_assess.style.format("{:.3f}")
    .highlight_max(axis=0, subset=["Recall", "F1"], color="#c6efce")
    .highlight_min(axis=0, subset=["Grade Error"], color="#c6efce"))

,% AEs Probed,Patient Comfort,Overall,After Round 1,After Round 2,After Round 3
Prompt Strategy,,,,,,
Baseline,0.637,0.489,0.321,0.479,0.583,0.625
Concise,0.683,0.518,0.367,0.438,0.500,0.667
Full Context,0.750,0.480,0.376,0.646,0.688,0.750
No Drug Info,0.696,0.511,0.355,0.438,0.583,0.667
Few-Shot,0.779,0.503,0.395,0.646,0.750,0.750
Deployment-Ready,0.812,0.513,0.421,0.354,0.750,0.792


,Recall,Precision,F1,Grade Error
Prompt Strategy,,,,
Baseline,0.500,0.198,0.192,0.250
Concise,0.479,0.198,0.211,0.250
Full Context,0.438,0.156,0.163,0.310
No Drug Info,0.646,0.229,0.271,0.380
Few-Shot,0.583,0.187,0.229,0.250
Deployment-Ready,0.688,0.291,0.317,0.250


In [11]:
display(HTML("<h3>How detection improves with each round of conversation</h3>"))
display(HTML("<p><em>% AEs Probed = fraction of ground-truth AEs the nurse has asked about by that round</em></p>"))

prog_rows = []
for internal, display_name in INTERNAL_TO_DISPLAY.items():
    ae = res[internal]["turn_ae_progression"]
    mood = res[internal]["turn_mood_progression"]
    prog_rows.append({
        "Strategy": display_name,
        "% AEs Probed (Round 1)": ae[0],
        "% AEs Probed (Round 2)": ae[1],
        "% AEs Probed (Round 3)": ae[2],
        "Patient Comfort (Round 1)": mood[0],
        "Patient Comfort (Round 2)": mood[1],
        "Patient Comfort (Round 3)": mood[2],
    })

df_prog = pd.DataFrame(prog_rows).set_index("Strategy")
display(df_prog.style.format("{:.3f}").background_gradient(cmap="YlGn", axis=None))

,% AEs Probed (Round 1),% AEs Probed (Round 2),% AEs Probed (Round 3),Patient Comfort (Round 1),Patient Comfort (Round 2),Patient Comfort (Round 3)
Strategy,,,,,,
Baseline,0.479,0.583,0.625,0.533,0.501,0.431
Concise,0.438,0.500,0.667,0.509,0.500,0.545
Full Context,0.646,0.688,0.750,0.509,0.466,0.466
No Drug Info,0.438,0.583,0.667,0.466,0.523,0.544
Few-Shot,0.646,0.750,0.750,0.540,0.534,0.433
Deployment-Ready,0.354,0.750,0.792,0.507,0.536,0.497


## 6. Conclusions

In [12]:
templates = list(INTERNAL_TO_DISPLAY.keys())
best_pareto_k = max(templates, key=lambda t: res[t]["avg_pareto"])
best_t4_k = max(templates, key=lambda t: res[t]["t4_f1"])
best_ae_k = max(templates, key=lambda t: res[t]["avg_ae"])

bp = INTERNAL_TO_DISPLAY[best_pareto_k]
bt = INTERNAL_TO_DISPLAY[best_t4_k]
ba = INTERNAL_TO_DISPLAY[best_ae_k]

findings = f"""
### Results

| What we measured | Best strategy | Score |
|---|---|---|
| Best at detecting AEs while keeping patient engaged | **{bp}** | Overall = {res[best_pareto_k]['avg_pareto']:.3f} |
| Most accurate final clinical assessment | **{bt}** | F1 = {res[best_t4_k]['t4_f1']:.3f} (Recall = {res[best_t4_k]['t4_recall']:.3f}) |
| Most AEs probed during conversation | **{ba}** | {res[best_ae_k]['avg_ae']:.1%} of ground-truth AEs asked about |

### Key Takeaways

1. **Deployment-Ready prompt is the best overall strategy** — it achieves the highest combined AE detection + patient comfort, using only information realistically available in production (drug profile, visual assessment, speaking style guidelines).

2. **More rounds = more AEs found** — Deployment-Ready detects {res['F_realistic']['turn_ae_progression'][0]:.0%} of AEs after Round 1 → {res['F_realistic']['turn_ae_progression'][2]:.0%} after Round 3, a {res['F_realistic']['turn_ae_progression'][2]/max(res['F_realistic']['turn_ae_progression'][0],0.01):.1f}× improvement through multi-round probing.

3. **Patient personality data is not needed** — Full Context (which includes unrealistic personality data) does not outperform Deployment-Ready, suggesting MedGemma can adapt its approach from conversational cues alone.

4. **Strong intrinsic medical knowledge** — even No Drug Info (no AE profile provided) achieves competitive detection, indicating MedGemma's pre-training covers drug-specific adverse event knowledge.

5. **Few-shot examples help conversation style but not final accuracy** — Few-Shot has high in-conversation AE targeting but similar assessment F1, suggesting the model already knows *what* to look for; examples mainly help *how* to ask.
"""
display(Markdown(findings))


### Results

| What we measured | Best strategy | Score |
|---|---|---|
| Best at detecting AEs while keeping patient engaged | **Deployment-Ready** | Overall = 0.421 |
| Most accurate final clinical assessment | **Deployment-Ready** | F1 = 0.317 (Recall = 0.688) |
| Most AEs probed during conversation | **Deployment-Ready** | 81.2% of ground-truth AEs asked about |

### Key Takeaways

1. **Deployment-Ready prompt is the best overall strategy** — it achieves the highest combined AE detection + patient comfort, using only information realistically available in production (drug profile, visual assessment, speaking style guidelines).

2. **More rounds = more AEs found** — Deployment-Ready detects 35% of AEs after Round 1 → 79% after Round 3, a 2.2× improvement through multi-round probing.

3. **Patient personality data is not needed** — Full Context (which includes unrealistic personality data) does not outperform Deployment-Ready, suggesting MedGemma can adapt its approach from conversational cues alone.

4. **Strong intrinsic medical knowledge** — even No Drug Info (no AE profile provided) achieves competitive detection, indicating MedGemma's pre-training covers drug-specific adverse event knowledge.

5. **Few-shot examples help conversation style but not final accuracy** — Few-Shot has high in-conversation AE targeting but similar assessment F1, suggesting the model already knows *what* to look for; examples mainly help *how* to ask.


In [13]:
del model
torch.cuda.empty_cache()
print("Model unloaded.")

Model unloaded.
